# SolarSDE Master Part 1 — Foundations + Training + Cross-Validation

**Target venue: Solar Energy (Elsevier, IF ~6, hybrid — no mandatory APC).**

**Kaggle workflow:** Run this notebook (~9-10h on P100), then save the version
as a Kaggle Dataset. Open 08b_master_part2_kaggle.ipynb and attach this run's
output as an input dataset. ~6-8h more. Final results zip downloads from 08b.

## What this notebook does

| Step | What | Time |
|------|------|------|
| Setup + soft fast-start | Pull cached Golden artifacts from GitHub if available | 5 min |
| Retrain Golden (conditional) | CloudCV download (~2.6 GB) + BMS + preprocess + VAE training (20 epochs) + latent extraction + physics features + extended splits | ~3-4h |
| Image features | Optical flow + sun-ROI + cloud fraction on Golden test/val/train | ~30 min |
| Train SolarSDE | CTI-gated Neural SDE + Score Decoder, seed 42 | ~3-4h |
| 5-fold leave-one-day-out CV | Reviewer-required validation across train days | ~2.5h |
| Zip outputs | Single zip + summary CSV in /kaggle/working/ | <5 min |

## Kaggle setup

1. Settings → Accelerator: **GPU P100** (or T4 if P100 unavailable)
2. Settings → Internet: **On**
3. Run all cells
4. When done: File → Save Version → Save & Run All (creates a Kaggle Dataset
   from /kaggle/working/)
5. Download `solarsde_outputs.zip` from the Output tab
6. Open 08b_master_part2_kaggle.ipynb. Add Data → Your Datasets → attach the
   version you just saved. Run 08b.


## 0. Setup (Kaggle)

In [ ]:
# ==== Kaggle setup (Part 1) ====
import os, sys
from pathlib import Path

IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
IN_COLAB = "google.colab" in sys.modules
assert IN_KAGGLE or not IN_COLAB, "This notebook is designed for Kaggle. Use 08_solarsde_master_colab.ipynb on Colab."

PERSIST_DIR = Path("/kaggle/working/solarsde_outputs") if IN_KAGGLE else (Path.cwd() / "solarsde_outputs")
WORK_DIR    = Path("/kaggle/working/solarsde") if IN_KAGGLE else (Path.cwd() / "solarsde_work")

for d in [PERSIST_DIR, WORK_DIR,
          PERSIST_DIR / "checkpoints", PERSIST_DIR / "results",
          PERSIST_DIR / "latents",     PERSIST_DIR / "splits",
          PERSIST_DIR / "extended",    PERSIST_DIR / "figures"]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR        = WORK_DIR / "data"
CHECKPOINT_DIR  = PERSIST_DIR / "checkpoints"
RESULTS_DIR     = PERSIST_DIR / "results"
LATENT_DIR      = PERSIST_DIR / "latents"
SPLITS_DIR      = PERSIST_DIR / "splits"
EXTENDED_DIR    = PERSIST_DIR / "extended"
FIGURES_DIR     = PERSIST_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Kaggle env: {IN_KAGGLE}  PERSIST_DIR: {PERSIST_DIR}  WORK_DIR: {WORK_DIR}")

def pip_install(*pkgs):
    import subprocess
    for p in pkgs:
        try: __import__(p.split("==")[0].replace("-", "_"))
        except ImportError:
            subprocess.run(["pip", "install", "-q", p], check=True)
pip_install("pvlib", "h5py", "scikit-learn", "scipy", "tqdm",
            "opencv-python-headless", "matplotlib", "pyarrow")

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import gc, json, time, shutil, requests
from tqdm import tqdm
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==== Soft fast-start: try to pull cached artifacts from GitHub (best-effort) ====
import requests
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

def gh_pull_soft(rel_path, dest):
    if dest.exists() and dest.stat().st_size > 100:
        return True
    try:
        r = requests.get(f"{GITHUB_RAW}/{rel_path}", timeout=180)
        if r.status_code == 200 and len(r.content) > 100:
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(r.content)
            return True
    except Exception:
        pass
    return False

required = {
    CHECKPOINT_DIR / "vae_best.pt":   "colab_outputs/checkpoints/vae_best.pt",
    SPLITS_DIR    / "train.parquet":  "colab_outputs/splits/train.parquet",
    SPLITS_DIR    / "val.parquet":    "colab_outputs/splits/val.parquet",
    SPLITS_DIR    / "test.parquet":   "colab_outputs/splits/test.parquet",
    EXTENDED_DIR  / "train.parquet":  "colab_outputs/extended/train.parquet",
    EXTENDED_DIR  / "val.parquet":    "colab_outputs/extended/val.parquet",
    EXTENDED_DIR  / "test.parquet":   "colab_outputs/extended/test.parquet",
}
for split in ["train", "val", "test"]:
    for key in ["latents", "cti", "ghi", "covariates", "is_ramp", "kt",
                "ghi_clearsky", "physics_features"]:
        required[LATENT_DIR / f"{split}_{key}.npy"] = f"colab_outputs/latents/{split}_{key}.npy"
optional = {
    CHECKPOINT_DIR / "sde_best.pt":   "colab_outputs/checkpoints/sde_best.pt",
    CHECKPOINT_DIR / "score_best.pt": "colab_outputs/checkpoints/score_best.pt",
}
n_have = 0
for dest, rel in {**required, **optional}.items():
    if gh_pull_soft(rel, dest): n_have += 1
print(f"Fast-start: {n_have} artifacts available locally or pulled from GitHub")

HAVE_VAE     = (CHECKPOINT_DIR / "vae_best.pt").exists()
HAVE_SPLITS  = (SPLITS_DIR / "train.parquet").exists()
HAVE_LATENTS = all((LATENT_DIR / f"{s}_latents.npy").exists() for s in ["train", "val", "test"])
HAVE_KT      = all((LATENT_DIR / f"{s}_kt.npy").exists() for s in ["train", "val", "test"])
HAVE_PHYS    = all((LATENT_DIR / f"{s}_physics_features.npy").exists() for s in ["train", "val", "test"])
HAVE_EXT     = (EXTENDED_DIR / "train.parquet").exists()
NEED_GOLDEN_RETRAIN = not (HAVE_VAE and HAVE_SPLITS and HAVE_LATENTS and HAVE_KT and HAVE_PHYS)
print(f"NEED_GOLDEN_RETRAIN = {NEED_GOLDEN_RETRAIN}")


## 1. Shared model definitions

In [ ]:
# ==== Shared model definitions (matches Notebooks 1 + 2) ====

# --- CS-VAE (needed only for sanity; not retrained here) ---
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([nn.Conv2d(in_ch, ch, 4, 2, 1),
                           nn.GroupNorm(min(32, ch), ch),
                           nn.SiLU(inplace=True)])
            in_ch = ch
        self.conv = nn.Sequential(*layers); self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

# --- Neural SDE ---
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c): return self.net(torch.cat([z, t, c], dim=-1))

SIGMA_FLOOR_BASE = 0.01

class CTIDiffNet(nn.Module):
    """v2: diffusion floor + CTI scaling. sigma = floor(1+10*cti) + learned_softplus"""
    def __init__(self, z_dim=64, h=64, sigma_floor=SIGMA_FLOOR_BASE):
        super().__init__()
        self.sigma_floor = sigma_floor
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        base_floor = self.sigma_floor * (1.0 + 10.0 * cti)
        learned = self.out(self.state(z) * self.cti_gate(cti))
        return base_floor + learned

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim; self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c); sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        # v2: log-space diffusion matching (well-conditioned, prevents sigma collapse)
        resid = (zn - z - mu * dt).pow(2) / dt + 1e-8
        log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
        return {"loss": drift_l + self.lambda_sigma * log_diff_l,
                "drift": drift_l, "diffusion": log_diff_l}

# --- Score Decoder v3 (RESIDUAL prediction: delta_kt = kt(t+h) - kt(t)) ---
#
# v2 predicted absolute kt(t+h). For stable conditions where kt(t+h) ≈ kt(t),
# the model had to learn a near-identity mapping — neural nets are bad at this.
#
# v3 predicts delta_kt = kt(t+h) - kt(t). Targets are concentrated near 0
# (most timesteps have small change). At sampling time, we add the sampled
# delta to the current kt to get the prediction:
#
#   kt(t+h)_predicted = kt(t)_observed + delta_kt_sampled
#   GHI(t+h) = kt(t+h)_predicted * ghi_clearsky(t+h)
#
# This is the persistence-anchored parameterization. Default behavior is
# "no change" (delta=0 = persistence). Model learns to deviate from
# persistence only when context says so.

GHI_SCALE = 1200.0
KT_SCALE = 1.5
DELTA_KT_SCALE = 1.0    # delta_kt typically in [-1.0, 1.0], rarely outside

class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        # Inputs: (noisy_target, s, z, cti, c, kt_current)
        d_in = 1 + 1 + z_dim + 1 + c_dim + 1
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c, kt_cur):
        return self.net(torch.cat([g, s, z, cti, c, kt_cur], dim=-1))

class CondScoreDecoder(nn.Module):
    """v3: predicts delta_kt with persistence anchoring.

    Default mode (predict_mode='delta'):
      target = kt(t+h) - kt(t)
      sample: kt(t+h) = kt(t) + delta_sampled
    Other modes (legacy):
      'kt'  : predicts kt(t+h) directly (v2)
      'ghi' : predicts GHI(t+h) directly (v1)
    """
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02,
                 predict_mode='delta'):
        super().__init__()
        self.steps = steps
        self.predict_mode = predict_mode
        if predict_mode == 'delta':
            self.target_scale = DELTA_KT_SCALE
        elif predict_mode == 'kt':
            self.target_scale = KT_SCALE
        else:
            self.target_scale = GHI_SCALE
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps); alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _normalize(self, y):
        # For delta, scale [-DELTA_KT_SCALE, DELTA_KT_SCALE] -> [-1, 1]
        if self.predict_mode == 'delta':
            return y.clamp(-self.target_scale, self.target_scale) / self.target_scale
        else:
            return y / self.target_scale * 2.0 - 1.0
    def _denormalize(self, y):
        if self.predict_mode == 'delta':
            return y * self.target_scale
        else:
            return (y + 1.0) / 2.0 * self.target_scale

    def training_loss(self, kt_target, kt_current, z, cti, c):
        """Train on residual (or absolute, depending on mode)."""
        if self.predict_mode == 'delta':
            target_raw = kt_target - kt_current
        elif self.predict_mode == 'kt':
            target_raw = kt_target
        else:
            target_raw = kt_target  # caller passes ghi values in this mode
        t_norm = self._normalize(target_raw)
        B = t_norm.shape[0]
        si = torch.randint(0, self.steps, (B,), device=t_norm.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(t_norm)
        ts = self.sac[si].unsqueeze(-1) * t_norm + self.s1mac[si].unsqueeze(-1) * eps
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        pred_noise = self.score(ts, sn, z, cti, c, kt_cur_in)
        return {"loss": F.mse_loss(pred_noise, eps)}

    @torch.no_grad()
    def sample(self, z, cti, c, kt_current, n=1):
        """Returns samples in kt-space.
           predict_mode='delta': returns kt(t+h) = kt_current + delta_sampled (clamped to [0, KT_SCALE])
           predict_mode='kt'   : returns kt(t+h) directly
           predict_mode='ghi'  : returns GHI(t+h) directly (caller doesn't multiply by gcs)
        """
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        kt_cur_in = kt_current.unsqueeze(-1) if kt_current.dim() == 1 else kt_current
        kt_cur_e = kt_cur_in.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            eps_pred = self.score(x, sn, z_e, cti_e, c_e, kt_cur_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x - b / (1 - ac).sqrt() * eps_pred)
            if i > 0: x = mean + b.sqrt() * torch.randn_like(x)
            else:     x = mean
        y_unscaled = self._denormalize(x)   # in target space (delta_kt or kt or ghi)
        if self.predict_mode == 'delta':
            kt_out = (kt_cur_e + y_unscaled).clamp(0.0, KT_SCALE)
        elif self.predict_mode == 'kt':
            kt_out = y_unscaled.clamp(0.0, KT_SCALE)
        else:
            kt_out = y_unscaled.clamp(0.0, GHI_SCALE)
        return kt_out.view(B, n)

# --- Metrics ---
def crps_empirical(y_true, y_samples):
    """y_true: (N,), y_samples: (N, M). Returns per-point CRPS (N,)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp_metric(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw_metric(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    if len(y_true) == 0: return {"crps": 0, "picp": 0, "pinaw": 0, "rmse": 0, "mae": 0, "ramp_crps": 0}
    y_med = np.median(y_samples, axis=1)
    y_range = float(y_true.max() - y_true.min())
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps":  float(crps.mean()),
        "picp":  picp_metric(y_true, y_samples, alpha),
        "pinaw": pinaw_metric(y_samples, y_range, alpha),
        "rmse":  float(np.sqrt(np.mean((y_true - y_med) ** 2))),
        "mae":   float(np.mean(np.abs(y_true - y_med))),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    else:
        out["ramp_crps"] = 0.0
    return out

# --- SDE solver (with stability clamping) ---
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD_RAW = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
Z_STD = torch.maximum(Z_STD_RAW, torch.full_like(Z_STD_RAW, 0.05))
Z_CLAMP_STDS = 8.0
MU_CAP = 10.0
SIGMA_CAP = 5.0
del _train_Z_np

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    """v4: with mixed-horizon training, the drift takes (z, normalized_horizon, c).
    At inference, we pass normalized_horizon = current_step / MAX_HORIZON as time input.
    This matches how the SDE was trained (drift(z, k/180, c) -> dz/k).
    The EM step uses physical dt=1.0; drift output is already in per-step units.
    """
    B, d = z0.shape
    mx = max(horizons); hset = set(horizons)
    MAX_HORIZON = 180.0
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t_norm = torch.full((B * N, 1), (step + 1) / MAX_HORIZON, device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t_norm, c_e, cti_e, dt)
        if (step + 1) in hset: out[step + 1] = z.view(B, N, d).clone()
    return out

print("Shared code loaded.")


## RETRAIN — Golden CO (skipped if cached)

In [ ]:
# ==== Conditional guards on the Golden retrain blocks below ====
# Each Notebook 1 block (CLOUDCV_DOWNLOAD/EXTRACT/BMS/PREPROCESS/VAE/LATENT)
# has its own internal "skip if output exists" check. The wrapper here just
# documents the intent and lets you skip the whole stage with one toggle.

ENABLE_GOLDEN_RETRAIN = NEED_GOLDEN_RETRAIN   # auto-detected by fast-start
if not ENABLE_GOLDEN_RETRAIN:
    print("[SKIP] Golden retrain disabled (all artifacts present).")


In [ ]:
LATENT_DIM = 64
IMG_SIZE = 128
# ==== CS-VAE model definition ====
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([
                nn.Conv2d(in_ch, ch, 4, 2, 1),
                nn.GroupNorm(min(32, ch), ch),
                nn.SiLU(inplace=True),
            ])
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(256, 128, 64, 32)):
        super().__init__()
        self.init_ch = channels[0]
        self.fc = nn.Linear(latent_dim, channels[0] * 8 * 8)
        layers = []
        for i in range(len(channels) - 1):
            layers.extend([
                nn.ConvTranspose2d(channels[i], channels[i+1], 4, 2, 1),
                nn.GroupNorm(min(32, channels[i+1]), channels[i+1]),
                nn.SiLU(inplace=True),
            ])
        layers.extend([nn.ConvTranspose2d(channels[-1], 3, 4, 2, 1), nn.Sigmoid()])
        self.deconv = nn.Sequential(*layers)
    def forward(self, z):
        h = self.fc(z).view(-1, self.init_ch, 8, 8)
        return self.deconv(h)

class CloudStateVAE(nn.Module):
    def __init__(self, latent_dim=64, beta=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.beta = beta
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def reparam(self, mu, lv):
        return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
    def forward(self, x):
        mu, lv = self.encoder(x)
        z = self.reparam(mu, lv)
        return self.decoder(z), mu, lv
    def loss(self, x, recon, mu, lv):
        rec = F.mse_loss(recon, x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return {"loss": rec + self.beta * kl, "recon": rec, "kl": kl}
    @torch.no_grad()
    def encode_mu(self, x):
        mu, _ = self.encoder(x)
        return mu

n_params = sum(p.numel() for p in CloudStateVAE().parameters())
print(f"CS-VAE parameters: {n_params:,}")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not all((DATA_DIR / 'cloudcv' / f).exists() for f in ['2019_09_07.tar.gz']):
    # ==== Download CloudCV dataset (with retry + size validation + S3 redirect handling) ====
    # NREL's data.nrel.gov endpoint redirects to data.nlr.gov which redirects to an
    # AWS S3 pre-signed URL valid for 5 minutes. If a download is interrupted, the
    # pre-signed URL may have expired by retry time, so each retry re-issues the
    # original request (which gets a fresh pre-signed URL).
    import requests, time
    from tqdm import tqdm

    CLOUDCV_DIR = DATA_DIR / "cloudcv"
    CLOUDCV_DIR.mkdir(parents=True, exist_ok=True)

    CLOUDCV_FILES = {
        "2019_09_07.tar.gz": "https://data.nrel.gov/system/files/248/1727737056-2019_09_07.tar.gz",
        "2019_09_08.tar.gz": "https://data.nrel.gov/system/files/248/1727737056-2019_09_08.tar.gz",
        "2019_09_14.tar.gz": "https://data.nrel.gov/system/files/248/1727737056-2019_09_14.tar.gz",
        "2019_09_15.tar.gz": "https://data.nrel.gov/system/files/248/1727737056-2019_09_15.tar.gz",
        "2019_09_21.tar.gz": "https://data.nrel.gov/system/files/248/1727737586-2019_09_21.tar.gz",
        "2019_09_22.tar.gz": "https://data.nrel.gov/system/files/248/1727737586-2019_09_22.tar.gz",
        "2019_09_28.tar.gz": "https://data.nrel.gov/system/files/248/1727737586-2019_09_28.tar.gz",
        "2019_09_29.tar.gz": "https://data.nrel.gov/system/files/248/1727737586-2019_09_29.tar.gz",
    }
    MIN_TAR_SIZE_BYTES = 100 * 1024 * 1024   # CloudCV daily tars are 300-400 MB; reject anything <100 MB as corrupt

    def download_with_retry(url, dest, min_size=MIN_TAR_SIZE_BYTES, max_retries=4):
        """Download with retry + size validation. Re-issues request each attempt
        so AWS pre-signed URLs are refreshed. Returns True if final file passes size check."""
        if dest.exists() and dest.stat().st_size >= min_size:
            print(f"  Already have: {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
            return True
        if dest.exists() and dest.stat().st_size < min_size:
            print(f"  Partial/corrupt file {dest.name} ({dest.stat().st_size / 1e6:.2f} MB) — removing.")
            dest.unlink()

        last_err = None
        for attempt in range(1, max_retries + 1):
            try:
                print(f"  [attempt {attempt}/{max_retries}] {dest.name}")
                with requests.get(url, stream=True, timeout=600, allow_redirects=True) as r:
                    r.raise_for_status()
                    total = int(r.headers.get("content-length", 0))
                    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pb:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk); pb.update(len(chunk))
                if dest.stat().st_size < min_size:
                    raise RuntimeError(f"downloaded size {dest.stat().st_size / 1e6:.2f} MB < min {min_size / 1e6:.0f} MB")
                return True
            except Exception as e:
                last_err = e
                print(f"    FAILED: {e}")
                if dest.exists():
                    dest.unlink()
                if attempt < max_retries:
                    wait = 5 * attempt   # 5s, 10s, 15s exponential-ish backoff
                    print(f"    waiting {wait}s before retry ...")
                    time.sleep(wait)
        print(f"  GAVE UP on {dest.name} after {max_retries} attempts. Last error: {last_err}")
        return False

    print("=" * 70)
    print("Downloading CloudCV dataset (8 days, ~2.6 GB)")
    print("=" * 70)
    failed = []
    for name, url in CLOUDCV_FILES.items():
        if not download_with_retry(url, CLOUDCV_DIR / name):
            failed.append(name)
    if failed:
        raise RuntimeError(f"CloudCV download incomplete: {failed}. Re-run this cell or check network.")
    print("CloudCV download complete (all 8 files verified > 100 MB).")


In [ ]:
if ENABLE_GOLDEN_RETRAIN:
    # ==== Extract CloudCV archives (with corruption detection + retry) ====
    import tarfile

    print("Extracting tar.gz archives ...")
    corrupt = []
    for tgz in sorted(CLOUDCV_DIR.glob("2019_*.tar.gz")):
        stem = tgz.stem.replace(".tar", "")
        out = CLOUDCV_DIR / stem
        if out.exists() and any(out.rglob("*.jpg")):
            n = sum(1 for _ in (out / "images").glob("*.jpg")) if (out / "images").exists() else 0
            if n > 0:
                print(f"  {stem}: already extracted ({n} images)")
                continue
        out.mkdir(parents=True, exist_ok=True)
        try:
            with tarfile.open(tgz, "r:gz") as tf:
                tf.extractall(out)
            n_imgs = sum(1 for _ in (out / "images").glob("*.jpg")) if (out / "images").exists() else 0
            if n_imgs < 100:
                raise RuntimeError(f"only {n_imgs} images extracted (expected ~500-1000)")
            print(f"  {stem}: extracted ({n_imgs} images)")
        except Exception as e:
            print(f"  {stem}: EXTRACT FAILED ({e}) — marking tar for re-download")
            corrupt.append(tgz.name)
            if tgz.exists():
                tgz.unlink()

    if corrupt:
        raise RuntimeError(f"Corrupt tarballs deleted: {corrupt}. Re-run CLOUDCV_DOWNLOAD cell.")

    # Free disk by removing tarballs after successful extraction
    for tgz in CLOUDCV_DIR.glob("*.tar.gz"):
        try: tgz.unlink()
        except Exception: pass
    print("Extraction complete. tar.gz archives removed to free disk.")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (DATA_DIR / 'bms' / 'bms_srrl_2019.csv').exists():
    # ==== Download BMS meteorological data (90-day period) ====
    BMS_DIR = DATA_DIR / "bms"
    BMS_DIR.mkdir(parents=True, exist_ok=True)
    BMS_PATH = BMS_DIR / "bms_srrl_2019.csv"

    BMS_URL = "https://midcdmz.nrel.gov/apps/data_api.pl?site=BMS&begin=20190905&end=20191203&inst=1&type=data"

    if BMS_PATH.exists() and BMS_PATH.stat().st_size > 10_000_000:
        print(f"BMS data already cached: {BMS_PATH.stat().st_size/1e6:.1f} MB")
    else:
        print(f"Downloading BMS 1-minute data from NREL MIDC API ...")
        r = requests.get(BMS_URL, timeout=600)
        r.raise_for_status()
        BMS_PATH.write_text(r.text)
        print(f"Saved: {BMS_PATH.stat().st_size/1e6:.1f} MB")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (SPLITS_DIR / 'train.parquet').exists():
    # ==== Preprocessing: parse CloudCV + BMS, align, compute features ====
    import numpy as np
    import pandas as pd
    from datetime import datetime
    import pvlib
    from pvlib.location import Location

    SRRL = Location(latitude=39.742, longitude=-105.18, tz="America/Denver",
                    altitude=1829, name="NREL SRRL")

    def parse_ts(s):
        s = s.strip()
        if s.startswith("UTC-7_"):
            s = s[6:]
        date_p, time_p = s.split("-")
        y, mo, d = date_p.split("_")
        tp = time_p.split("_")
        h, mi, sec = tp[0], tp[1], tp[2]
        us = tp[3] if len(tp) > 3 else "0"
        return datetime(int(y), int(mo), int(d), int(h), int(mi), int(sec), int(us))

    def load_cloudcv_day(day_dir):
        csv = day_dir / "pyranometer.csv"
        imgs = day_dir / "images"
        if not csv.exists():
            return pd.DataFrame()
        rows = []
        for line in open(csv):
            line = line.strip()
            if not line or "," not in line:
                continue
            parts = line.split(",")
            if len(parts) < 2:
                continue
            try:
                ts = parse_ts(parts[0])
            except Exception:
                continue
            mv = float(parts[1].strip())
            img_name = parts[0].strip() + ".jpg"
            img_path = imgs / img_name
            rows.append({
                "timestamp": ts, "millivolts": mv,
                "image_path": str(img_path),
                "image_exists": img_path.exists(),
            })
        df = pd.DataFrame(rows)
        if len(df) > 0:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df

    print("Loading CloudCV days ...")
    day_dirs = sorted([d for d in CLOUDCV_DIR.iterdir() if d.is_dir() and d.name.startswith("2019")])
    all_dfs = []
    for d in day_dirs:
        df = load_cloudcv_day(d)
        if len(df) > 0:
            all_dfs.append(df)
            print(f"  {d.name}: {len(df)} rows ({df['image_exists'].sum()} images)")

    cloudcv = pd.concat(all_dfs, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    print(f"Total CloudCV rows: {len(cloudcv)}")

    print("\nLoading BMS ...")
    bms_raw = pd.read_csv(BMS_PATH)
    print(f"  BMS rows: {len(bms_raw)}")
    print(f"  BMS columns: {len(bms_raw.columns)}")

    # Build BMS timestamps from Year, DOY, MST
    ts_list = []
    for _, r in bms_raw.iterrows():
        try:
            y, doy, mst = int(r["Year"]), int(r["DOY"]), int(r["MST"])
            dt = datetime.strptime(f"{y}-{doy}", "%Y-%j").replace(hour=mst // 60, minute=mst % 60)
            ts_list.append(dt)
        except Exception:
            ts_list.append(pd.NaT)
    bms_raw["timestamp"] = pd.to_datetime(ts_list)

    bms = pd.DataFrame({
        "timestamp": bms_raw["timestamp"],
        "ghi_bms": bms_raw.get("Global LI-200 [W/m^2]"),
        "dni_bms": bms_raw.get("Direct NIP [W/m^2]"),
        "dhi_bms": bms_raw.get("Diffuse CM22-1 (vent/cor) [W/m^2]"),
        "temperature": bms_raw.get("Deck Dry Bulb Temp [deg C]"),
        "humidity": bms_raw.get("Deck RH [%]"),
        "wind_speed": bms_raw.get("Avg Wind Speed @ 19ft [m/s]"),
        "pressure": bms_raw.get("Station Pressure [mBar]"),
        "cloud_cover_total": bms_raw.get("Total Cloud Cover [%]"),
    })
    for c in bms.columns:
        if c != "timestamp":
            bms[c] = pd.to_numeric(bms[c], errors="coerce").replace([-7999, -6999, -9999], np.nan)

    print(f"  BMS GHI range: [{bms['ghi_bms'].min():.1f}, {bms['ghi_bms'].max():.1f}] W/m²")

    # Interpolate BMS GHI to 10s
    print("\nInterpolating BMS GHI to 10-second resolution ...")
    bms_ghi = bms[["timestamp", "ghi_bms"]].dropna().copy()
    bms_ghi = bms_ghi.set_index("timestamp").sort_index()
    bms_10s = bms_ghi.resample("10s").interpolate(method="linear")

    cloudcv["ts_round"] = cloudcv["timestamp"].dt.round("10s")
    ghi_vals = []
    for ts in cloudcv["ts_round"]:
        if ts in bms_10s.index:
            ghi_vals.append(float(bms_10s.loc[ts, "ghi_bms"]))
        else:
            i = bms_10s.index.get_indexer([ts], method="nearest")[0]
            ghi_vals.append(float(bms_10s.iloc[i]["ghi_bms"]) if 0 <= i < len(bms_10s) else np.nan)
    cloudcv["ghi"] = np.clip(ghi_vals, 0, None)

    # Merge meteo covariates
    cloudcv["ts_minute"] = cloudcv["timestamp"].dt.floor("min")
    bms["ts_minute"] = bms["timestamp"].dt.floor("min")
    merged = cloudcv.merge(bms.drop(columns=["timestamp"]), on="ts_minute", how="left")
    for c in ["temperature", "humidity", "wind_speed", "pressure", "cloud_cover_total"]:
        if c in merged.columns:
            merged[c] = merged[c].ffill().fillna(0)

    # Solar geometry + clear sky
    print("\nComputing solar geometry + clear-sky ...")
    tz_ts = pd.DatetimeIndex(merged["timestamp"]).tz_localize("America/Denver")
    solpos = SRRL.get_solarposition(tz_ts)
    merged["solar_zenith"] = solpos["apparent_zenith"].values
    cs = SRRL.get_clearsky(tz_ts, model="ineichen")
    merged["ghi_clearsky"] = cs["ghi"].values
    with np.errstate(divide="ignore", invalid="ignore"):
        kt = merged["ghi"].values / merged["ghi_clearsky"].values
        kt = np.where(merged["ghi_clearsky"].values < 10, 0.0, kt)
    merged["clear_sky_index"] = np.clip(kt, 0, 1.5)

    # Quality filter: daytime, image exists, valid GHI
    before = len(merged)
    merged = merged[(merged["solar_zenith"] <= 85.0) & (merged["ghi"] >= 0)
                    & (merged["ghi"].notna()) & (merged["image_exists"])].reset_index(drop=True)
    print(f"Quality filter: {before} -> {len(merged)} rows")

    # Ramp detection (|ΔGHI| > 50 W/m² in 60s)
    dg = merged["ghi"].diff(6).abs() / 1.0
    merged["is_ramp"] = (dg > 50.0).fillna(False)
    print(f"Ramp events: {int(merged['is_ramp'].sum())} ({merged['is_ramp'].mean()*100:.1f}%)")

    # Chronological split (5 train / 1 val / 2 test by date)
    dates = sorted(merged["timestamp"].dt.date.unique())
    print(f"\nUnique dates: {len(dates)}")
    n_tr = max(1, int(len(dates) * 0.625))
    n_val = max(1, int(len(dates) * 0.125))
    train_dates = set(dates[:n_tr])
    val_dates = set(dates[n_tr:n_tr + n_val])
    test_dates = set(dates[n_tr + n_val:])

    train_df = merged[merged["timestamp"].dt.date.isin(train_dates)].reset_index(drop=True)
    val_df   = merged[merged["timestamp"].dt.date.isin(val_dates)].reset_index(drop=True)
    test_df  = merged[merged["timestamp"].dt.date.isin(test_dates)].reset_index(drop=True)

    SPLITS_DIR = PERSIST_DIR / "splits"
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    train_df.to_parquet(SPLITS_DIR / "train.parquet")
    val_df.to_parquet(SPLITS_DIR / "val.parquet")
    test_df.to_parquet(SPLITS_DIR / "test.parquet")

    print(f"\nSplit sizes:")
    print(f"  train: {len(train_df):>6} rows ({len(train_dates)} days)")
    print(f"  val:   {len(val_df):>6} rows ({len(val_dates)} days)")
    print(f"  test:  {len(test_df):>6} rows ({len(test_dates)} days)")
    print(f"  GHI range overall: [{merged['ghi'].min():.1f}, {merged['ghi'].max():.1f}] W/m²")


In [ ]:
if ENABLE_GOLDEN_RETRAIN:
    # ==== Image Dataset (loads JPEGs on the fly) ====
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    import numpy as np
    import pandas as pd

    def load_img(path, size=128):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        side = min(w, h)
        l, t = (w - side) // 2, (h - side) // 2
        img = img.crop((l, t, l + side, t + side)).resize((size, size), Image.BILINEAR)
        return np.array(img, dtype=np.float32) / 255.0

    class SkyImageDataset(Dataset):
        def __init__(self, parquet_path, size=128):
            self.df = pd.read_parquet(parquet_path)
            if "image_exists" in self.df.columns:
                self.df = self.df[self.df["image_exists"]].reset_index(drop=True)
            self.size = size
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            p = self.df.iloc[i]["image_path"]
            arr = load_img(p, self.size)
            return torch.from_numpy(arr).permute(2, 0, 1)

    for sp in ["train", "val", "test"]:
        ds = SkyImageDataset(SPLITS_DIR / f"{sp}.parquet")
        print(f"  {sp}: {len(ds)} images")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (CHECKPOINT_DIR / 'vae_best.pt').exists():
    # ==== Train CS-VAE ====
    from tqdm import tqdm
    import time

    IMG_SIZE = 128
    LATENT_DIM = 64
    BATCH = 32
    EPOCHS = 20           # trimmed from 100 — sufficient for 128x128 with this dataset size
    LR = 1e-4
    SEED = 42

    torch.manual_seed(SEED); np.random.seed(SEED)

    train_ds = SkyImageDataset(SPLITS_DIR / "train.parquet", size=IMG_SIZE)
    val_ds   = SkyImageDataset(SPLITS_DIR / "val.parquet",   size=IMG_SIZE)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

    model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    print(f"Training VAE: {EPOCHS} epochs, batch={BATCH}, img={IMG_SIZE}x{IMG_SIZE}, latent_dim={LATENT_DIM}")
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    print("=" * 70)

    best_val = float("inf")
    history = []
    t_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tl, tr, tk = 0, 0, 0
        for img in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            img = img.to(DEVICE, non_blocking=True)
            rec, mu, lv = model(img)
            losses = model.loss(img, rec, mu, lv)
            opt.zero_grad(); losses["loss"].backward(); opt.step()
            tl += losses["loss"].item(); tr += losses["recon"].item(); tk += losses["kl"].item()
        tl /= len(train_loader); tr /= len(train_loader); tk /= len(train_loader)

        model.eval()
        vl = 0
        with torch.no_grad():
            for img in val_loader:
                img = img.to(DEVICE, non_blocking=True)
                rec, mu, lv = model(img)
                vl += model.loss(img, rec, mu, lv)["loss"].item()
        vl /= max(len(val_loader), 1)

        elapsed = (time.time() - t_start) / 60
        print(f"Epoch {epoch:3d}/{EPOCHS} | train={tl:.4f} (rec={tr:.4f}, kl={tk:.4f}) | val={vl:.4f} | {elapsed:.1f} min")
        history.append({"epoch": epoch, "train_loss": tl, "train_recon": tr, "train_kl": tk, "val_loss": vl})

        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_best.pt")
            print(f"  [best] saved checkpoint (val={vl:.4f})")

    torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_final.pt")
    pd.DataFrame(history).to_csv(RESULTS_DIR / "vae_training_history.csv", index=False)
    print("=" * 70)
    print(f"VAE training complete. Best val loss: {best_val:.4f}. Total time: {(time.time()-t_start)/60:.1f} min")


In [ ]:
if ENABLE_GOLDEN_RETRAIN and not (LATENT_DIR / 'test_latents.npy').exists():
    # ==== Extract latents + compute CTI ====
    from torch.utils.data import DataLoader

    def cti_from_latents(Z, window=10):
        """Compute CTI as L2 norm of variance of latent velocity over a sliding window."""
        T = Z.shape[0]
        cti = np.zeros(T, dtype=np.float32)
        for t in range(window, T):
            win = Z[t - window:t]
            v = np.diff(win, axis=0)
            var = v.var(axis=0)
            cti[t] = np.linalg.norm(var, ord=2)
        return cti

    # Load best VAE
    model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / "vae_best.pt", map_location=DEVICE))
    model.eval()

    print("Extracting latents for train / val / test ...")
    for split in ["train", "val", "test"]:
        ds = SkyImageDataset(SPLITS_DIR / f"{split}.parquet", size=IMG_SIZE)
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
        all_mu = []
        with torch.no_grad():
            for img in tqdm(loader, desc=f"Encoding {split}"):
                img = img.to(DEVICE, non_blocking=True)
                all_mu.append(model.encode_mu(img).cpu().numpy())
        Z = np.concatenate(all_mu, axis=0).astype(np.float32)
        cti = cti_from_latents(Z, window=10)

        df = ds.df
        ghi = df["ghi"].values.astype(np.float32)
        cov_cols = [c for c in ["solar_zenith", "clear_sky_index", "temperature",
                                "humidity", "wind_speed"] if c in df.columns]
        cov = df[cov_cols].fillna(0).values.astype(np.float32) if cov_cols else np.zeros((len(df), 0), np.float32)

        np.save(LATENT_DIR / f"{split}_latents.npy", Z)
        np.save(LATENT_DIR / f"{split}_cti.npy",     cti)
        np.save(LATENT_DIR / f"{split}_ghi.npy",     ghi)
        np.save(LATENT_DIR / f"{split}_covariates.npy", cov)
        np.save(LATENT_DIR / f"{split}_is_ramp.npy", df["is_ramp"].values.astype(bool))

        print(f"  {split}: Z={Z.shape}, CTI range=[{cti.min():.4f}, {cti.max():.4f}], "
              f"GHI range=[{ghi.min():.1f}, {ghi.max():.1f}], covariates={cov.shape}")
    print("Latent extraction complete.")


In [ ]:
# ==== Save kt + ghi_clearsky + physics features for each split ====
# Required by LOAD_DATA: kt, ghi_clearsky come from the parquet columns.
# Physics features (15-dim) are computed inline from the split parquets.

import numpy as np, pandas as pd

ALL_KT_DONE = all((LATENT_DIR / f"{s}_kt.npy").exists() and
                  (LATENT_DIR / f"{s}_ghi_clearsky.npy").exists()
                  for s in ["train", "val", "test"])
if ALL_KT_DONE:
    print("[SKIP] kt + ghi_clearsky already saved.")
else:
    print("Saving kt + ghi_clearsky from split parquets ...")
    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        kt = df["clear_sky_index"].values.astype(np.float32)
        gcs = df["ghi_clearsky"].values.astype(np.float32)
        np.save(LATENT_DIR / f"{split}_kt.npy", kt)
        np.save(LATENT_DIR / f"{split}_ghi_clearsky.npy", gcs)
        print(f"  {split}: kt range [{kt.min():.3f}, {kt.max():.3f}], gcs range [{gcs.min():.1f}, {gcs.max():.1f}]")

ALL_PHYS_DONE = all((LATENT_DIR / f"{s}_physics_features.npy").exists()
                    for s in ["train", "val", "test"])
if ALL_PHYS_DONE:
    print("[SKIP] Physics features already computed.")
else:
    print("\nComputing physics features (15-dim per row) ...")
    def compute_physics_15(df):
        df = df.reset_index(drop=True).copy()
        n = len(df)
        zenith = df["solar_zenith"].values.astype(np.float32)
        zenith_clip = np.clip(zenith, 0, 89.9)
        zenith_rad = np.deg2rad(zenith_clip)
        air_mass = 1.0 / (np.cos(zenith_rad) + 0.50572 * (96.07995 - zenith_clip) ** -1.6364)
        air_mass = np.clip(air_mass, 1.0, 40.0).astype(np.float32)
        zenith_rate = np.zeros(n, dtype=np.float32)
        zenith_rate[1:] = (zenith[1:] - zenith[:-1]) / 10.0
        az = df.get("solar_azimuth", pd.Series(np.zeros(n))).values.astype(np.float32)
        az_rad = np.deg2rad(az)
        ts = pd.to_datetime(df["timestamp"])
        hour_frac = (ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0).values
        doy = ts.dt.dayofyear.values
        kt = df["clear_sky_index"].values.astype(np.float32)
        def trend(s, lag):
            o = np.zeros_like(s); o[lag:] = (s[lag:] - s[:-lag]) / lag; return o
        def rstd(s, w):
            o = np.zeros_like(s)
            for i in range(w, len(s)):
                o[i] = np.std(s[i-w:i])
            return o
        ghi = df["ghi"].values.astype(np.float32)
        pyr = df["millivolts"].values.astype(np.float32) if "millivolts" in df.columns else np.zeros(n, np.float32)
        return np.stack([
            air_mass, zenith_rate,
            np.sin(az_rad).astype(np.float32), np.cos(az_rad).astype(np.float32),
            np.sin(2*np.pi*hour_frac/24).astype(np.float32),
            np.cos(2*np.pi*hour_frac/24).astype(np.float32),
            np.sin(2*np.pi*doy/365.25).astype(np.float32),
            np.cos(2*np.pi*doy/365.25).astype(np.float32),
            np.clip((hour_frac - 6.0) / 12.0, 0, 1).astype(np.float32),
            trend(kt, 6), trend(kt, 30), trend(kt, 60),
            rstd(kt, 30).astype(np.float32),
            (rstd(ghi, 6) / 1200.0).astype(np.float32),
            pyr,
        ], axis=1)
    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        feats = compute_physics_15(df)
        np.save(LATENT_DIR / f"{split}_physics_features.npy", feats)
        print(f"  {split}: physics features shape={feats.shape}")


In [ ]:
# ==== Build extended (90-day BMS-only) splits for LSTM baselines ====
# These provide ~12x more training data for the LSTM/MC-Dropout/Deep-Ensemble/
# TimeGrad baselines. Without them, baselines train on only ~5 days of 10s data.

if HAVE_EXTENDED:
    print("[SKIP] Extended splits already present.")
else:
    print("Building extended (90-day BMS-only) splits ...")
    import pandas as pd, numpy as np

    BMS_PATH = DATA_DIR / "bms" / "bms_srrl_2019.csv"
    if not BMS_PATH.exists():
        print("[WARN] BMS data not present — extended splits cannot be built.")
        print("       Re-run BMS_DOWNLOAD cell first, or LSTM baselines will be limited.")
    else:
        try:
            import pvlib
            from pvlib.location import Location
        except ImportError:
            pip_install("pvlib")
            import pvlib
            from pvlib.location import Location
        SRRL = Location(latitude=39.742, longitude=-105.18, tz="America/Denver",
                        altitude=1829, name="NREL SRRL")

        bms_raw = pd.read_csv(BMS_PATH)
        from datetime import datetime
        ts_list = []
        for _, r in bms_raw.iterrows():
            try:
                y, doy, mst = int(r["Year"]), int(r["DOY"]), int(r["MST"])
                hh, mm = divmod(mst, 100)
                dt = datetime.strptime(f"{y}-{doy}", "%Y-%j").replace(hour=hh, minute=mm)
                ts_list.append(dt)
            except Exception:
                ts_list.append(pd.NaT)
        bms_raw["timestamp"] = pd.to_datetime(ts_list)
        bms_raw = bms_raw.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

        cols_map = {
            "ghi": "Global LI-200 [W/m^2]",
            "dni": "Direct NIP [W/m^2]",
            "dhi": "Diffuse CM22-1 (vent/cor) [W/m^2]",
            "temperature": "Deck Dry Bulb Temp [deg C]",
            "humidity":    "Deck RH [%]",
            "wind_speed":  "Avg Wind Speed @ 19ft [m/s]",
            "millivolts":  "Global LI-200 [W/m^2]",
        }
        out = pd.DataFrame({"timestamp": bms_raw["timestamp"]})
        for k, src in cols_map.items():
            out[k] = pd.to_numeric(bms_raw.get(src), errors="coerce").replace(
                [-7999, -6999, -9999], np.nan)
        out["ghi"] = out["ghi"].clip(lower=0)
        out = out.dropna(subset=["ghi"]).reset_index(drop=True)

        # Solar geometry + clear sky
        tz_ts = pd.DatetimeIndex(out["timestamp"]).tz_localize("America/Denver")
        sp = SRRL.get_solarposition(tz_ts)
        out["solar_zenith"] = sp["apparent_zenith"].values
        out["solar_azimuth"] = sp["azimuth"].values
        cs = SRRL.get_clearsky(tz_ts, model="ineichen")
        out["ghi_clearsky"] = cs["ghi"].values
        out["clear_sky_index"] = np.clip(out["ghi"] / out["ghi_clearsky"].replace(0, np.nan), 0, 1.5).fillna(0)
        out = out[out["solar_zenith"] <= 85.0].reset_index(drop=True)
        out["is_ramp"] = (out["ghi"].diff(1).abs() > 50.0).fillna(False)

        # Chronological 60/15/15 split by date
        dates = sorted(out["timestamp"].dt.date.unique())
        n_tr = int(len(dates) * 0.7); n_val = int(len(dates) * 0.15)
        tr_set = set(dates[:n_tr]); va_set = set(dates[n_tr:n_tr+n_val])
        ext_tr = out[out["timestamp"].dt.date.isin(tr_set)].reset_index(drop=True)
        ext_va = out[out["timestamp"].dt.date.isin(va_set)].reset_index(drop=True)
        ext_te = out[~out["timestamp"].dt.date.isin(tr_set | va_set)].reset_index(drop=True)
        ext_tr.to_parquet(EXTENDED_DIR / "train.parquet")
        ext_va.to_parquet(EXTENDED_DIR / "val.parquet")
        ext_te.to_parquet(EXTENDED_DIR / "test.parquet")
        print(f"  extended: train={len(ext_tr):,}  val={len(ext_va):,}  test={len(ext_te):,}")


## 2. Load data tensors

In [ ]:
# ==== Load all data tensors (tolerant: degrades gracefully if extended missing) ====
def load_split(s):
    orig_cov = np.load(LATENT_DIR / f"{s}_covariates.npy")
    phys = np.load(LATENT_DIR / f"{s}_physics_features.npy")
    img_feat_path = LATENT_DIR / f"{s}_image_features.npy"
    if img_feat_path.exists():
        img_feats = np.load(img_feat_path)
        cov = np.concatenate([orig_cov, phys, img_feats], axis=1).astype(np.float32)
    else:
        cov = np.concatenate([orig_cov, phys], axis=1).astype(np.float32)
    return {
        "Z":    np.load(LATENT_DIR / f"{s}_latents.npy"),
        "cti":  np.load(LATENT_DIR / f"{s}_cti.npy"),
        "ghi":  np.load(LATENT_DIR / f"{s}_ghi.npy"),
        "cov":  cov,
        "ramp": np.load(LATENT_DIR / f"{s}_is_ramp.npy"),
        "kt":   np.load(LATENT_DIR / f"{s}_kt.npy"),
        "gcs":  np.load(LATENT_DIR / f"{s}_ghi_clearsky.npy"),
    }
data = {s: load_split(s) for s in ["train", "val", "test"]}
print(f"\n  Covariate dim: {data['train']['cov'].shape[1]}  "
      f"(5 original + 15 physics + "
      f"{data['train']['cov'].shape[1] - 20} image features)")
for s, d in data.items():
    print(f"  {s}: Z={d['Z'].shape}, GHI=[{d['ghi'].min():.0f},{d['ghi'].max():.0f}], ramps={int(d['ramp'].sum())}")

train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df   = pd.read_parquet(SPLITS_DIR / "val.parquet")
test_df  = pd.read_parquet(SPLITS_DIR / "test.parquet")
print(f"\n8-day image splits: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")

# Extended (90-day BMS) splits — used by LSTM/MC-Dropout/TimeGrad/Deep-Ensemble baselines.
# If missing, we fall back to using the regular train_df/val_df for those baselines.
HAVE_EXT = (EXTENDED_DIR / "train.parquet").exists() and (EXTENDED_DIR / "val.parquet").exists()
if HAVE_EXT:
    ext_train = pd.read_parquet(EXTENDED_DIR / "train.parquet")
    ext_val   = pd.read_parquet(EXTENDED_DIR / "val.parquet")
    print(f"90-day extended:    train={len(ext_train):,} val={len(ext_val):,}")
else:
    print("[WARN] Extended (90-day BMS) parquets missing — LSTM baselines will train on the")
    print("       8-day image splits instead, with reduced sample count.")
    # Fallback: replicate the structure expected by BASELINES_CODE
    ext_train = train_df.copy()
    ext_val   = val_df.copy()

Z_DIM = data["train"]["Z"].shape[1]
C_DIM = max(1, data["train"]["cov"].shape[1])
print(f"\nZ_DIM={Z_DIM}, C_DIM={C_DIM}")

HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
# Larger N_EVAL gives tighter bootstrap CIs. 2000 is ~12% of typical test set,
# enough for ramp events to be represented at expected ~5-10% rate.
N_EVAL = min(2000, len(data["test"]["Z"]) - max(HORIZONS) - 1)
SEQ_LEN = 30
print(f"Horizons: {list(HORIZON_MIN.values())} min, MC samples: {N_SAMPLES}, N_EVAL: {N_EVAL}")


## STAGE B — Image features (optical flow + sun-ROI + cloud fraction)

In [ ]:
# ==== STAGE -1: Image feature extraction (optical flow + sun-ROI + cloud fraction) ====
# This is OPTIONAL — if image_features.npy is present, we skip. Otherwise we either:
#   a) Download the 8 days of CloudCV images from NREL (2.6 GB) + extract features
#   b) Skip gracefully if download fails (features default to zero)
#
# On Kaggle/Colab this adds ~30-45 min to the run but gives large expected
# improvement on ramp events and during cloud transitions.

STAGE_M1_OUT = LATENT_DIR / "test_image_features.npy"
if STAGE_M1_OUT.exists():
    print(f"[SKIP] Stage -1 done (image features exist).")
else:
    print("=" * 70)
    print("STAGE -1: Extracting image features (optical flow + sun-ROI + cloud fraction)")
    print("=" * 70)
    pip_install("opencv-python-headless")
    import cv2
    from PIL import Image
    import tarfile

    RAW_DIR = WORK_DIR / "cloudcv"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CLOUDCV_FILES = {
        "2019_09_07.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_07.tar.gz",
        "2019_09_08.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_08.tar.gz",
        "2019_09_14.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_14.tar.gz",
        "2019_09_15.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_15.tar.gz",
        "2019_09_21.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_21.tar.gz",
        "2019_09_22.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_22.tar.gz",
        "2019_09_28.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_28.tar.gz",
        "2019_09_29.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_29.tar.gz",
    }

    # Download + extract
    all_days_present = all((RAW_DIR / fn.replace(".tar.gz", "") / "pyranometer.csv").exists()
                           for fn in CLOUDCV_FILES)
    if not all_days_present:
        print("Downloading CloudCV archives ...")
        for name, url in CLOUDCV_FILES.items():
            tgz = RAW_DIR / name
            day_dir = RAW_DIR / name.replace(".tar.gz", "")
            if (day_dir / "pyranometer.csv").exists():
                continue
            if not tgz.exists():
                r = requests.get(url, stream=True, timeout=600)
                with open(tgz, "wb") as f:
                    for chunk in r.iter_content(chunk_size=65536): f.write(chunk)
                print(f"  downloaded {name}  ({tgz.stat().st_size/1e6:.0f} MB)")
            day_dir.mkdir(parents=True, exist_ok=True)
            with tarfile.open(tgz, "r:gz") as tf: tf.extractall(day_dir)
            tgz.unlink()   # free disk
            print(f"  extracted {day_dir.name}")

    IMG_SIZE = 128
    def load_img_small(path):
        img = Image.open(path).convert("RGB")
        w, h = img.size; side = min(w, h)
        l, t = (w-side)//2, (h-side)//2
        img = img.crop((l, t, l+side, t+side)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return np.array(img, dtype=np.uint8)

    def sun_px(zen, az, size=IMG_SIZE):
        r_frac = np.clip(zen / 90.0, 0, 1)
        rp = r_frac * (size / 2 - 5)
        a = np.deg2rad(az)
        dx = rp * np.sin(a); dy = -rp * np.cos(a)
        return int(size // 2 + dx), int(size // 2 + dy)

    def sun_roi(gray, sx, sy, r=12):
        H, W = gray.shape
        x0,x1 = max(0,sx-r), min(W,sx+r); y0,y1 = max(0,sy-r), min(H,sy+r)
        if x1<=x0 or y1<=y0: return 0.0, 0.0, 0.0
        roi = gray[y0:y1, x0:x1].astype(np.float32)
        b = roi.mean()/255.0; v = roi.var()/(255.0**2)
        gx = cv2.Sobel(roi, cv2.CV_32F, 1, 0); gy = cv2.Sobel(roi, cv2.CV_32F, 0, 1)
        e = np.sqrt(gx**2 + gy**2).mean()/255.0
        return float(b), float(v), float(e)

    # Helper: fix image_path to point to downloaded location
    def fix_path(orig_path):
        fn = Path(orig_path).name
        for day in RAW_DIR.iterdir():
            if day.is_dir():
                p = day / "images" / fn
                if p.exists(): return str(p)
        return orig_path

    for split in ["train", "val", "test"]:
        df = pd.read_parquet(SPLITS_DIR / f"{split}.parquet")
        if "image_exists" in df.columns:
            df = df[df["image_exists"]].reset_index(drop=True)
        n = len(df)
        feats = np.zeros((n, 10), dtype=np.float32)
        prev_gray = None
        print(f"Processing {split} ({n} rows) ...")
        for i in tqdm(range(n), desc=f"  {split}"):
            row = df.iloc[i]
            img_path = fix_path(row["image_path"])
            if not Path(img_path).exists():
                prev_gray = None; continue
            try:
                img = load_img_small(img_path)
                gray = np.mean(img, axis=2).astype(np.uint8)
            except Exception:
                prev_gray = None; continue
            # Optical flow
            if prev_gray is not None:
                try:
                    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                                       0.5, 3, 15, 3, 5, 1.2, 0)
                    fx = flow[..., 0].mean(); fy = flow[..., 1].mean()
                    mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
                    fmag = mag.mean(); fvar = mag.var()
                    fdir = np.arctan2(fy, fx)
                    feats[i, 0] = fx / 10.0; feats[i, 1] = fy / 10.0
                    feats[i, 2] = fmag / 10.0
                    feats[i, 3] = np.sin(fdir); feats[i, 4] = np.cos(fdir)
                    feats[i, 5] = np.tanh(fvar / 100.0)
                except Exception: pass
            # Sun ROI
            sx, sy = sun_px(float(row["solar_zenith"]), float(row.get("solar_azimuth", 0.0)))
            b, v, e = sun_roi(gray, sx, sy, r=12)
            feats[i, 6] = b; feats[i, 7] = v; feats[i, 8] = e
            # Cloud fraction
            feats[i, 9] = float((img.mean(axis=2) / 255.0 > 0.75).mean())
            prev_gray = gray

        np.save(LATENT_DIR / f"{split}_image_features.npy", feats)
        print(f"  saved {split}_image_features.npy  shape={feats.shape}  "
              f"mean={feats.mean():.3f}  std={feats.std():.3f}")

    # Clean up raw images to save disk
    import shutil
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    print("Image features extracted. Raw images removed to free disk.")
    print("NOTE: re-load `data` below to pick up new image features in covariates.")

    # Reload data with image features included
    data = {s: load_split(s) for s in ["train", "val", "test"]}
    C_DIM = max(1, data["train"]["cov"].shape[1])
    print(f"C_DIM updated to {C_DIM} (with image features)")


## STAGE C — Train SolarSDE on Golden (auto-resume)

In [ ]:
# ==== STAGE 0: Train SDE + Score Decoder if missing (was Notebook 2) ====
if not NEED_NB2_TRAINING:
    print("[SKIP] Stage 0 — SDE + Score checkpoints already present.")
else:
    print("=" * 70)
    print("STAGE 0: Training Neural SDE + Score Decoder (inline)")
    print("=" * 70)

    class LatentSeqDataset(Dataset):
        def __init__(self, d):
            self.Z=d["Z"]; self.cti=d["cti"]; self.ghi=d["ghi"]; self.cov=d["cov"]
            self.kt=d["kt"]; self.gcs=d["gcs"]
        def __len__(self): return max(0, len(self.Z) - 1)
        def __getitem__(self, i):
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "z_next": torch.from_numpy(self.Z[i+1]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "ghi": torch.tensor(float(self.ghi[i])),
                    "kt":  torch.tensor(float(self.kt[i])),
                    "gcs": torch.tensor(float(self.gcs[i])),
                    "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM)}

    tr_ds_s0 = LatentSeqDataset(data["train"])
    va_ds_s0 = LatentSeqDataset(data["val"])

    # ---- Train SDE with MIXED-HORIZON training + ramp oversampling (v4) ----
    print("\n[S0a] Training Neural SDE v4 (mixed-horizon, ramp-oversampled) ...")
    torch.manual_seed(42); np.random.seed(42)
    sde0 = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM).to(DEVICE)
    opt = torch.optim.Adam(sde0.parameters(), lr=1e-4)

    class MixedHorizonDataset(Dataset):
        def __init__(self, d, horizon_choices=(1,5,10,30,60,90,120,180), seed=42):
            self.Z=d["Z"]; self.cti=d["cti"]; self.cov=d["cov"]; self.ramp=d["ramp"]
            self.hs=horizon_choices; self.max_h=max(horizon_choices)
            self.rng=np.random.default_rng(seed)
        def __len__(self): return max(0, len(self.Z) - self.max_h)
        def __getitem__(self, i):
            k = int(self.rng.choice(self.hs))
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "z_next": torch.from_numpy(self.Z[i+k]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "cov": torch.from_numpy(self.cov[i]).float(),
                    "k":   torch.tensor(float(k))}

    mh_tr = MixedHorizonDataset(data["train"])
    mh_va = MixedHorizonDataset(data["val"])

    # Ramp oversampling (5x weight on rows containing a ramp within max horizon)
    ramp_window = np.zeros(len(mh_tr), dtype=np.float32)
    tr_ramp = data["train"]["ramp"]
    for i in range(len(mh_tr)):
        end = min(i + mh_tr.max_h, len(tr_ramp))
        ramp_window[i] = 1.0 if tr_ramp[i:end].any() else 0.0
    weights = np.where(ramp_window > 0, 5.0, 1.0).astype(np.float32)
    from torch.utils.data import WeightedRandomSampler
    sampler = WeightedRandomSampler(weights=weights.tolist(),
                                    num_samples=len(mh_tr), replacement=True)
    dl_tr = DataLoader(mh_tr, batch_size=128, sampler=sampler, drop_last=True, num_workers=0)
    dl_va = DataLoader(mh_va, batch_size=128, shuffle=False, num_workers=0)
    print(f"  MixedHorizonDataset: {len(mh_tr)} rows, ramp-in-window fraction = "
          f"{ramp_window.mean()*100:.1f}% (weighted 5x)")

    EPOCHS_SDE = 150
    best_val = float("inf"); t0 = time.time(); hist = []
    for ep in range(1, EPOCHS_SDE + 1):
        sde0.train(); tl = td = ts = 0; n = 0
        for b in dl_tr:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
            # Normalized horizon as time input
            t = (b["k"].float().unsqueeze(-1) / 180.0).to(DEVICE)
            dt_k = b["k"].float().unsqueeze(-1).to(DEVICE)
            mu = sde0.drift(z, t, c); sigma = sde0.diffusion(z, cti)
            dz_per_k = (zn - z) / dt_k
            drift_l = F.mse_loss(mu, dz_per_k)
            resid = (zn - z - mu * dt_k).pow(2) / dt_k + 1e-8
            log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
            loss = drift_l + sde0.lambda_sigma * log_diff_l
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(sde0.parameters(), 1.0); opt.step()
            tl += loss.item(); td += drift_l.item(); ts += log_diff_l.item(); n += 1
        tl /= n; td /= n; ts /= n
        sde0.eval(); vl = vn = 0
        with torch.no_grad():
            for b in dl_va:
                z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
                cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
                t = (b["k"].float().unsqueeze(-1) / 180.0).to(DEVICE)
                dt_k = b["k"].float().unsqueeze(-1).to(DEVICE)
                mu = sde0.drift(z, t, c); sigma = sde0.diffusion(z, cti)
                dz_per_k = (zn - z) / dt_k
                drift_l = F.mse_loss(mu, dz_per_k)
                resid = (zn - z - mu * dt_k).pow(2) / dt_k + 1e-8
                log_diff_l = F.mse_loss(torch.log(sigma.pow(2) + 1e-8), torch.log(resid))
                vl += (drift_l + sde0.lambda_sigma * log_diff_l).item(); vn += 1
        vl /= max(vn, 1)
        hist.append({"epoch": ep, "train_loss": tl, "drift": td, "diffusion": ts, "val_loss": vl})
        if ep % 15 == 0 or ep == 1:
            print(f"  SDE ep {ep:3d}/{EPOCHS_SDE} | train={tl:.5f} | val={vl:.5f} | {(time.time()-t0)/60:.1f}min")
        if vl < best_val:
            best_val = vl; torch.save(sde0.state_dict(), SDE_CKPT)
    pd.DataFrame(hist).to_csv(RESULTS_DIR / "sde_training_history.csv", index=False)
    print(f"  SDE done. Best val: {best_val:.6f}. Time: {(time.time()-t0)/60:.1f} min")

    # ---- Train Score Decoder (v2: predicts k_t = GHI/GHI_clearsky) ----
    print("\n[S0b] Training Score Decoder v3 (150 ep, cosine LR, target=delta_kt) ...")
    print("  v3 predicts kt(t+h) - kt(t) [persistence-anchored residual].")
    print("  Default behavior: 'no change' (delta=0 = persistence baseline).")
    print("  Model only has to learn cloud-driven deviations.")
    torch.manual_seed(42)
    score0 = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, predict_mode='delta').to(DEVICE)
    opt_s = torch.optim.Adam(score0.parameters(), lr=2e-4)
    EPOCHS_SCORE = 150
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=EPOCHS_SCORE, eta_min=1e-5)
    # Training pairs + ramp oversampling for score decoder
    class TrainPairsDataset(Dataset):
        def __init__(self, d):
            self.Z=d["Z"]; self.cti=d["cti"]; self.cov=d["cov"]; self.kt=d["kt"]; self.ramp=d["ramp"]
        def __len__(self): return max(0, len(self.Z) - 1)
        def __getitem__(self, i):
            return {"z_t": torch.from_numpy(self.Z[i]).float(),
                    "cti": torch.tensor(float(self.cti[i])),
                    "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM),
                    "kt_current": torch.tensor(float(self.kt[i])),
                    "kt_target":  torch.tensor(float(self.kt[i+1]))}
    tp_tr = TrainPairsDataset(data["train"]); tp_va = TrainPairsDataset(data["val"])

    # Ramp oversampling (weight ramp rows 5x)
    tr_ramp_arr = data["train"]["ramp"][:len(tp_tr)]
    weights_s = np.where(tr_ramp_arr, 5.0, 1.0).astype(np.float32)
    sampler_s = WeightedRandomSampler(weights=weights_s.tolist(),
                                      num_samples=len(tp_tr), replacement=True)
    dl_tr_s = DataLoader(tp_tr, batch_size=256, sampler=sampler_s, drop_last=True)
    dl_va_s = DataLoader(tp_va, batch_size=256, shuffle=False)
    print(f"  Score decoder training: ramp rows weighted 5x "
          f"({tr_ramp_arr.sum()} ramp / {len(tp_tr)} total)")
    best_val = float("inf"); t0 = time.time(); hist = []
    for ep in range(1, EPOCHS_SCORE + 1):
        score0.train(); tl = 0; n = 0
        for b in dl_tr_s:
            z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
            c = b["cov"].to(DEVICE)
            kt_tgt = b["kt_target"].to(DEVICE).unsqueeze(-1)
            kt_cur = b["kt_current"].to(DEVICE).unsqueeze(-1)
            l = score0.training_loss(kt_tgt, kt_cur, z, cti, c)["loss"]
            opt_s.zero_grad(); l.backward()
            torch.nn.utils.clip_grad_norm_(score0.parameters(), 1.0)
            opt_s.step(); tl += l.item(); n += 1
        tl /= n; sched.step()
        score0.eval(); vl = vn = 0
        with torch.no_grad():
            for b in dl_va_s:
                z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
                c = b["cov"].to(DEVICE)
                kt_tgt = b["kt_target"].to(DEVICE).unsqueeze(-1)
                kt_cur = b["kt_current"].to(DEVICE).unsqueeze(-1)
                vl += score0.training_loss(kt_tgt, kt_cur, z, cti, c)["loss"].item(); vn += 1
        vl /= max(vn, 1)
        hist.append({"epoch": ep, "train_loss": tl, "val_loss": vl, "lr": opt_s.param_groups[0]["lr"]})
        if ep % 10 == 0 or ep == 1:
            print(f"  Score ep {ep:3d}/{EPOCHS_SCORE} | train={tl:.4f} | val={vl:.4f} | lr={opt_s.param_groups[0]['lr']:.2e} | {(time.time()-t0)/60:.1f}min")
        if vl < best_val:
            best_val = vl; torch.save(score0.state_dict(), SCORE_CKPT)
    pd.DataFrame(hist).to_csv(RESULTS_DIR / "score_training_history.csv", index=False)
    print(f"  Score done. Best val: {best_val:.4f}. Time: {(time.time()-t0)/60:.1f} min")

    # ---- Run main evaluation (reconstruct GHI = k_t_sampled * ghi_clearsky(t+h)) ----
    print("\n[S0c] Running main SolarSDE evaluation at all horizons ...")
    sde0.load_state_dict(torch.load(SDE_CKPT, map_location=DEVICE, weights_only=False)); sde0.eval()
    score0.load_state_dict(torch.load(SCORE_CKPT, map_location=DEVICE, weights_only=False)); score0.eval()
    te = data["test"]; res_s = {}
    for h in HORIZONS:
        yt, ys, rm = [], [], []
        for i in tqdm(range(0, N_EVAL, 32), desc=f"  h={HORIZON_MIN[h]}min"):
            idx = list(range(i, min(i + 32, N_EVAL)))
            z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
            c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
            cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
            kt_cur = torch.from_numpy(te["kt"][idx]).float().unsqueeze(-1).to(DEVICE)
            gcs_future = np.array([te["gcs"][ii + h] if (ii + h) < len(te["gcs"]) else 0.0
                                   for ii in idx], dtype=np.float32)
            with torch.no_grad():
                endp = solve_sde_horizons(sde0, z0, [h], c, cti, N=N_SAMPLES)[h]
                B, N, d = endp.shape
                kt_samples = score0.sample(endp.view(B*N, d),
                                 cti.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 c.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 kt_cur.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1),
                                 n=1).squeeze(-1).view(B, N).cpu().numpy()
                ghi_samples = kt_samples * gcs_future[:, None]
            for k, ii in enumerate(idx):
                j = ii + h
                if j < len(te["ghi"]):
                    yt.append(te["ghi"][j]); ys.append(ghi_samples[k]); rm.append(te["ramp"][j])
        m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
        m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
        res_s[h] = m
        print(f"    CRPS={m['crps']:.2f}  RMSE={m['rmse']:.2f}  PICP={m['picp']:.3f}  PINAW={m['pinaw']:.3f}")
    pd.DataFrame.from_dict(res_s, orient="index").sort_values("horizon_min").to_csv(
        RESULTS_DIR / "solar_sde_main_results.csv", index=False)
    print("\n[S0] STAGE 0 COMPLETE.")
    del sde0, score0, tr_ds_s0, va_ds_s0
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## STAGE CV — Leave-one-day-out cross-validation (reviewer requirement)

In [ ]:
# ==== Leave-one-day-out cross-validation across train days ====
# Strongest reviewer answer to "only 5 days?!" — show generalization across
# day-level holdouts. We share the VAE across folds (unsupervised, no label
# leakage) and retrain only the SDE + Score Decoder per fold.
#
# Per fold: ~30-40 min on A100, ~1.5h on T4. 5 folds = ~2.5-7.5 hours total.

CV_OUT = RESULTS_DIR / "cv_results.csv"
if CV_OUT.exists():
    print(f"[SKIP] CV already done -> {CV_OUT}")
    cv_summary = pd.read_csv(CV_OUT)
    print(cv_summary.to_string(index=False))
else:
    print("=" * 70)
    print("LEAVE-ONE-DAY-OUT CROSS VALIDATION")
    print("=" * 70)

    # Identify unique training days
    tr_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
    if "image_exists" in tr_df.columns:
        tr_df = tr_df[tr_df["image_exists"]].reset_index(drop=True)
    tr_df["date"] = pd.to_datetime(tr_df["timestamp"]).dt.date
    days = sorted(tr_df["date"].unique())
    print(f"Train days: {[str(d) for d in days]}")

    # Indices of each day in the full train tensor arrays
    day_idx = {d: tr_df.index[tr_df["date"] == d].tolist() for d in days}

    # Load all latents/cov/etc.
    z_all   = np.load(LATENT_DIR / "train_latents.npy")
    cti_all = np.load(LATENT_DIR / "train_cti.npy")
    kt_all  = np.load(LATENT_DIR / "train_kt.npy")
    cov_all = np.concatenate([
        np.load(LATENT_DIR / "train_covariates.npy"),
        np.load(LATENT_DIR / "train_physics_features.npy"),
    ] + ([np.load(LATENT_DIR / "train_image_features.npy")]
         if (LATENT_DIR / "train_image_features.npy").exists() else []), axis=1).astype(np.float32)
    ghi_all = np.load(LATENT_DIR / "train_ghi.npy")
    gcs_all = np.load(LATENT_DIR / "train_ghi_clearsky.npy")

    fold_rows = []
    for fold_i, holdout_day in enumerate(days):
        print(f"\n--- Fold {fold_i + 1}/{len(days)}: holding out {holdout_day} ---")
        tr_mask = np.zeros(len(z_all), dtype=bool)
        for d in days:
            if d != holdout_day:
                for i in day_idx[d]: tr_mask[i] = True
        te_mask = ~tr_mask

        z_tr_fold, z_te_fold = z_all[tr_mask], z_all[te_mask]
        cti_tr_fold, cti_te_fold = cti_all[tr_mask], cti_all[te_mask]
        kt_tr_fold, kt_te_fold = kt_all[tr_mask], kt_all[te_mask]
        cov_tr_fold, cov_te_fold = cov_all[tr_mask], cov_all[te_mask]
        ghi_te_fold = ghi_all[te_mask]
        gcs_te_fold = gcs_all[te_mask]

        print(f"  train={len(z_tr_fold)}  test (held-out day)={len(z_te_fold)}")
        if len(z_te_fold) < 200:
            print(f"  [SKIP] held-out day has too few samples")
            continue

        # Train SDE on this fold (reduced epochs for speed)
        torch.manual_seed(42 + fold_i)
        np.random.seed(42 + fold_i)

        class MHDS_CV(Dataset):
            def __init__(self, z, cti, c, hs=(1, 5, 10, 30, 60, 90, 120, 180), seed=42):
                self.z = z; self.cti = cti; self.c = c
                self.hs = hs; self.rng = np.random.RandomState(seed)
                self.maxh = max(hs); self.idx = np.arange(len(z) - self.maxh)
            def __len__(self): return len(self.idx)
            def __getitem__(self, i):
                ii = self.idx[i]; k = int(self.rng.choice(self.hs))
                return {"z_t": torch.from_numpy(self.z[ii]),
                        "z_next": torch.from_numpy(self.z[ii + k]),
                        "k": torch.tensor(k, dtype=torch.float32),
                        "cti_t": torch.tensor(self.cti[ii], dtype=torch.float32),
                        "c_t": torch.from_numpy(self.c[ii])}
        mh = MHDS_CV(z_tr_fold, cti_tr_fold, cov_tr_fold, seed=42 + fold_i)
        dl = DataLoader(mh, batch_size=512, shuffle=True, num_workers=2,
                        pin_memory=True, drop_last=True)
        sde_cv = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM).to(DEVICE)
        opt = torch.optim.Adam(sde_cv.parameters(), lr=5e-4)
        for ep in range(1, 21):   # 20 epochs (vs 30 for main model)
            sde_cv.train(); tl = 0; n = 0
            for b in dl:
                z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
                k = b["k"].float().unsqueeze(-1).to(DEVICE); t = k / 180.0
                cti = b["cti_t"].unsqueeze(-1).to(DEVICE); c = b["c_t"].to(DEVICE)
                mu = sde_cv.drift(z, t, c); sigma = sde_cv.diffusion(z, cti)
                dz = (zn - z) / k
                drift_l = F.mse_loss(mu, dz)
                resid = zn - z - mu * k
                tv = (resid ** 2) / k.clamp(min=1.0)
                sq = sigma.pow(2).clamp(min=1e-6)
                diff_l = F.mse_loss(torch.log(sq + 1e-8), torch.log(tv + 1e-8))
                loss = drift_l + 0.5 * diff_l
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(sde_cv.parameters(), 1.0); opt.step()
                tl += loss.item(); n += 1
            if ep % 5 == 0: print(f"    SDE fold{fold_i+1} ep {ep}/20: loss={tl/n:.4f}")

        # Train score decoder
        class SDS_CV(Dataset):
            def __init__(self, z, cti, c, kt, hs=(1, 5, 10, 30, 60, 90, 120, 180), seed=42):
                self.z = z; self.cti = cti; self.c = c; self.kt = kt
                self.hs = hs; self.rng = np.random.RandomState(seed); self.maxh = max(hs)
            def __len__(self): return len(self.z) - self.maxh
            def __getitem__(self, i):
                k = int(self.rng.choice(self.hs))
                return {"kt_target": torch.tensor(self.kt[i + k], dtype=torch.float32),
                        "kt_current": torch.tensor(self.kt[i], dtype=torch.float32),
                        "z_t": torch.from_numpy(self.z[i]),
                        "cti_t": torch.tensor(self.cti[i], dtype=torch.float32),
                        "c_t": torch.from_numpy(self.c[i])}
        score_cv = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, predict_mode='delta').to(DEVICE)
        opt2 = torch.optim.Adam(score_cv.parameters(), lr=1e-4)
        sds = SDS_CV(z_tr_fold, cti_tr_fold, cov_tr_fold, kt_tr_fold, seed=42 + fold_i)
        sdl = DataLoader(sds, batch_size=512, shuffle=True, num_workers=2,
                         pin_memory=True, drop_last=True)
        for ep in range(1, 21):
            score_cv.train(); tl = 0; n = 0
            for b in sdl:
                loss_d = score_cv.training_loss(
                    b["kt_target"].unsqueeze(-1).to(DEVICE),
                    b["kt_current"].unsqueeze(-1).to(DEVICE),
                    b["z_t"].to(DEVICE), b["cti_t"].unsqueeze(-1).to(DEVICE),
                    b["c_t"].to(DEVICE))
                loss = loss_d["loss"] if isinstance(loss_d, dict) else loss_d
                opt2.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(score_cv.parameters(), 1.0); opt2.step()
                tl += loss.item(); n += 1
            if ep % 5 == 0: print(f"    Score fold{fold_i+1} ep {ep}/20: loss={tl/n:.4f}")

        # Eval on held-out day
        sde_cv.eval(); score_cv.eval()
        for h in HORIZONS:
            preds_l, truths_l = [], []
            n_eval_fold = len(z_te_fold) - h - 1
            for i in range(0, n_eval_fold, 32):
                end = min(i + 32, n_eval_fold); bs = end - i
                z0 = torch.from_numpy(z_te_fold[i:end]).to(DEVICE)
                z0 = z0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, Z_DIM)
                cti0 = torch.from_numpy(cti_te_fold[i:end]).unsqueeze(-1).to(DEVICE)
                cti0 = cti0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, 1)
                c0 = torch.from_numpy(cov_te_fold[i:end]).to(DEVICE)
                c0 = c0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, C_DIM)
                kt0 = torch.from_numpy(kt_te_fold[i:end]).unsqueeze(-1).to(DEVICE)
                kt0 = kt0.unsqueeze(1).repeat(1, N_SAMPLES, 1).reshape(-1, 1)
                with torch.no_grad():
                    z = z0
                    for s in range(h):
                        t = torch.full((bs * N_SAMPLES, 1), s / 180.0, device=DEVICE)
                        z = z + sde_cv.drift(z, t, c0) + sde_cv.diffusion(z, cti0) * torch.randn_like(z)
                    kt_pred = score_cv.sample(z, cti0, c0, kt0, n=1).squeeze(-1).cpu().numpy()
                    kt_pred = kt_pred.reshape(bs, N_SAMPLES)
                ghi_pred = kt_pred * gcs_te_fold[i:end][:, None]
                preds_l.append(ghi_pred); truths_l.append(ghi_te_fold[i + h:end + h])
            preds = np.concatenate(preds_l, axis=0); yt = np.concatenate(truths_l)
            crps = float(crps_empirical(yt, preds).mean())
            rmse = float(np.sqrt(((preds.mean(1) - yt) ** 2).mean()))
            picp = float(((np.percentile(preds, 5, axis=1) <= yt) &
                          (yt <= np.percentile(preds, 95, axis=1))).mean())
            fold_rows.append({"fold": fold_i + 1, "holdout_day": str(holdout_day),
                              "horizon_min": HORIZON_MIN[h], "crps": crps,
                              "rmse": rmse, "picp": picp, "n_eval": len(yt)})
            print(f"    h={HORIZON_MIN[h]:2d}min: CRPS={crps:.2f} RMSE={rmse:.2f} PICP={picp:.3f}")
        del sde_cv, score_cv, mh, dl, sds, sdl; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    cv_df = pd.DataFrame(fold_rows)
    cv_df.to_csv(RESULTS_DIR / "cv_results_per_fold.csv", index=False)

    # Aggregate: mean ± std across folds per horizon
    cv_summary = cv_df.groupby("horizon_min").agg(
        crps_mean=("crps", "mean"), crps_std=("crps", "std"),
        rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
        picp_mean=("picp", "mean"), picp_std=("picp", "std"),
        n_folds=("fold", "count"),
    ).reset_index()
    cv_summary.to_csv(CV_OUT, index=False)
    print(f"\n5-fold CV summary (mean ± std across folds):")
    for _, r in cv_summary.iterrows():
        print(f"  h={int(r['horizon_min']):2d}min: CRPS = {r['crps_mean']:.2f} ± {r['crps_std']:.2f}, "
              f"RMSE = {r['rmse_mean']:.2f} ± {r['rmse_std']:.2f}, "
              f"PICP = {r['picp_mean']:.3f} ± {r['picp_std']:.3f}")


## Final — Zip Part 1 outputs to /kaggle/working/

In [ ]:
# ==== Zip outputs and clean up to a single file for easy Kaggle download ====
import shutil

# Set this to False if you want to keep intermediate files for debugging
MINIMAL_OUTPUT = True

zip_path = Path("/kaggle/working/solarsde_outputs.zip") if IN_KAGGLE else (WORK_DIR / "solarsde_outputs.zip")
if zip_path.exists():
    zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path.name} ...")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"  Archive size: {size_mb:.1f} MB")

# Print summary table of contents before optional cleanup
print("\n" + "=" * 70)
print("ALL STAGES COMPLETE")
print("=" * 70)
summary_rows = []
for sub in ["splits", "extended", "checkpoints", "latents", "results", "figures"]:
    p = PERSIST_DIR / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {sub}/: {n} files, {total/1e6:.1f} MB")
        summary_rows.append({"folder": sub, "files": n, "size_mb": total / 1e6})

# Save a tiny summary CSV alongside the zip — useful for a quick peek without unzipping
summary_csv = (Path("/kaggle/working") if IN_KAGGLE else WORK_DIR) / "solarsde_outputs_summary.csv"
import pandas as pd
pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)

if MINIMAL_OUTPUT and IN_KAGGLE:
    # Clean up: delete the unzipped PERSIST_DIR and the raw-data WORK_DIR.
    # The zip contains everything from PERSIST_DIR. WORK_DIR holds raw downloads
    # (CloudCV tarballs, SKIPP'D HDF5, BMS CSV) which are regeneratable.
    print(f"\nCleaning intermediate files (MINIMAL_OUTPUT=True) ...")
    try:
        shutil.rmtree(PERSIST_DIR, ignore_errors=True)
        print(f"  removed {PERSIST_DIR.name}/ (contents archived in zip)")
    except Exception as e:
        print(f"  could not remove PERSIST_DIR: {e}")
    try:
        shutil.rmtree(WORK_DIR, ignore_errors=True)
        print(f"  removed {WORK_DIR.name}/ (raw downloads)")
    except Exception as e:
        print(f"  could not remove WORK_DIR: {e}")
    # List what remains in /kaggle/working/
    remaining = list(Path("/kaggle/working").iterdir())
    print(f"\nFinal /kaggle/working/ contents ({len(remaining)} entries):")
    for f in sorted(remaining):
        size = f.stat().st_size / 1e6
        print(f"  {f.name}  ({size:.1f} MB)" if f.is_file() else f"  {f.name}/")

if IN_COLAB:
    from google.colab import files
    try: files.download(str(zip_path))
    except Exception as e: print(f"Auto-download failed: {e}. File at {zip_path}")
elif IN_KAGGLE:
    print(f"\nDownload the zip from the Output tab on the right sidebar.")
    print(f"Or 'Save Version' to commit /kaggle/working/ as a Kaggle Dataset for the next notebook.")
else:
    print(f"\nLocal: file at {zip_path}")
